# 02.1 First Model: Exercises

## Contents

**Tools:** CADET-GUI widgets: unit parameters and simulation; CADET-Process (Python).

**Python used:** Parameter changes, copied model code, variable names and simple calculations.

**Practises:** `LumpedRateModelWithoutPores`, scale-up, effects of flow rate and porosity, and serial columns.

1. Column scale-up
2. Flow rate and porosity
3. Two columns in series (optional)

```{tip}
Run code cells from top to bottom with **Shift+Enter**. After entering, pasting or changing code, run that cell again to apply it, then rerun any later cells that use its results. Setup cells may show no output; they create objects for later cells. Fill empty answer cells as you go.
```

Use the tracer column to relate breakthrough time to liquid volume and flow. Exercises marked **optional** extend the main tasks and can be skipped.

## Starter column

```{note}
The supplied column has length 10 cm, diameter 2 cm, porosity 0.4 and dispersion `1e-7` m²/s. Feed is 1 mol/m³ tracer at 6 mL/min; cycle time is 1200 s.
```

```{tip}
Run the cell below once with **Shift+Enter** to set up the supplied model for this exercise.
```

In [ ]:
# Show plots inside the notebook, then load the shared model setup and editor.
%matplotlib inline
%run ../resources/first_steps/lesson_setup.py

from CADETProcess.simulator import Cadet
from cadetgui import SimulationWidget
import numpy as np

# Load a fresh 10 × 2 cm tracer column at 6 mL/min for these exercises.
tracer_component_system, tracer_units, tracer_sheet, tracer_process = tracer_exercise_model()

# Keep the starter retention time as a reference during the flow-rate trials.
t_R = 0.4 * np.pi * 0.02**2 * 0.1 / (4 * 1e-7)

```{tip}
Run the widget cell with **Shift+Enter**. Select `column` or `inlet` under **Parameters of** to change the setting named in the task, then click **Run simulation**. Select `outlet` under **Show** to compare breakthrough curves.
```

In [ ]:
tracer_configuration_widget(tracer_process, tracer_units).display()
tracer_simulation = SimulationWidget(tracer_process, markers={"starter t_R": t_R})
tracer_simulation.display()

```{tip}
**Copy the model setup:** use **Show code** in the **unit editor**, the panel with the **Parameters of** selector. Copy the **whole code block**, including the imports, components, units, connections and process setup, ending with `process.cycle_time = ...`.

The simulation panel also has **Show code**; that contains only the simulation and plotting calls.
```

## Exercise 1: scale up

The dashed `starter t_R` line marks the original retention time (about 126 s) and stays fixed while you change the model.

Use the retention-time formula: $t_R=\varepsilon_t\pi d^2L/(4Q)$.

1. Simulate the starter column. Note when the outlet reaches half the feed concentration.
2. Select `column` and change `diameter` from `0.02` to `0.04` m (2 to 4 cm). Keep the other column settings unchanged. Look at the $t_R$ formula: what flow rate would give the same $t_R$ again? Select `inlet`, enter your prediction in `flow_rate` (m³/s), and click **Run simulation**. Does the breakthrough match the `starter t_R` line?
3. Repeat the same experiment in Python. Set column `diameter` back to `0.02` m and inlet `flow_rate` to `1e-7` m³/s (6 mL/min). Click **Show code** in the **unit editor (Parameters of)** and paste the **complete model setup** into the answer cell. Keep the exported variable names (`component_system`, `inlet`, `column`, `outlet`, `flow_sheet` and `process`) and run it. Then run the supplied simulation and plotting cell: it repeats the diameter and flow changes from the GUI and overlays both curves.

In [ ]:
# Paste the model setup from Show code in the unit editor


Compare the supplied simulation and plot with the GUI result.

In [ ]:
# Simulate the copied baseline, then change diameter and flow rate in Python.
simulator = Cadet()
small = simulator.simulate(process)
column.diameter = 0.04
inlet.flow_rate = 4e-7
large = simulator.simulate(process)
# Each curve is labelled by its component, "Tracer"; name and colour them by column instead.
fig, ax = small.solution.outlet.outlet.plot()
ax.lines[-1].set(label="2 cm column, 6 mL/min", color="C0")
_ = large.solution.outlet.outlet.plot(ax=ax)
ax.lines[-1].set(label="4 cm column, 24 mL/min", color="C1", linestyle="--")
# Python plots show time in minutes.
_ = ax.axvline(t_R / 60, color="gray", linestyle=":", label="starter t_R")
_ = ax.set_xlim(0, 6)
_ = ax.legend(loc="center left", bbox_to_anchor=(1, 0.5))  # Legend beside the plot.

```{dropdown} Answer
The baseline liquid volume is 12.57 mL, so $t_R \approx 126$ s. Doubling the diameter multiplies the area by four: use 24 mL/min. With the same length, porosity, velocity and dispersion, the two curves overlap.
```

## Exercise 2: what moves the breakthrough?

Start from the starter settings: column `diameter` `0.02` m, `total_porosity` `0.4` and inlet `flow_rate` `1e-7` m³/s (6 mL/min). Simulate them once, then change one parameter at a time and simulate again:

| Parameter | Starter | Changed value |
|---|---|---|
| Flow rate | 6 mL/min = `1e-7` m³/s | 3 mL/min = `5e-8` m³/s |
| Porosity | 0.4 | 0.3 |

```{note}
CADET uses SI units. To convert a flow rate: 1 mL/min = 10⁻⁶ m³ / 60 s ≈ `1.67e-8` m³/s.
```

Which change delays the front? Compare with $t_R=\varepsilon_tV/Q$.

Now repeat the comparison in Python. With the starter settings in the editor, click **Show code** in the **unit editor (Parameters of)** and paste the **complete model setup** into the answer cell. Below it, also copy the simulation and plotting lines from the Exercise 1 cell (`simulator = Cadet()` up to the plot calls). Then change flow rate and porosity one at a time, simulating after each change and adding each curve to the same plot with `ax=ax`.

```{tip}
Every curve is labelled with its component name, `Tracer`. To tell them apart, rename and colour the curve you just plotted, e.g. `ax.lines[-1].set(label="3 mL/min", color="C1")`, and finish with `ax.legend()`.
```

```{warning}
A Python name points to the object assigned to it last. If you paste the setup unchanged, `column`, `process` and the other names now refer to the new model, and the model from **Exercise 1** is no longer reachable under them. If you later run the Exercise 1 cell again, the names point to a fresh Exercise 1 model again: what a name refers to depends on which cell ran last, not on its position in the notebook.

You can choose:

- **Keep the names:** simplest; the Exercise 2 model replaces the one from Exercise 1.
- **Rename:** both models stay available, which is good practice in your own projects. Add the prefix `comparison_` to every variable name, wherever it is used:

| Copied name | Renamed |
|---|---|
| `component_system` | `comparison_component_system` |
| `inlet` | `comparison_inlet` |
| `column` | `comparison_column` |
| `outlet` | `comparison_outlet` |
| `flow_sheet` | `comparison_flow_sheet` |
| `process` | `comparison_process` |

If you rename, also use the new names in the copied simulation and plotting lines, e.g. `simulator.simulate(comparison_process)`. Quoted names such as `"column"` can stay: they name units inside the new flow sheet.
```

In [ ]:
# Paste the model setup from Show code, then simulate and plot both changes


```{dropdown} Answer
Halving flow doubles the residence time. Reducing porosity reduces the liquid volume and brings the front forward. A late curve alone cannot identify which parameter is wrong.
```

## Exercise 3 (optional): two columns

Start again from the starter settings in the editor. Click **Show code** in the **unit editor (Parameters of)** and paste the **complete model setup** into the answer cell. You may keep the exported names this time or rename them as in Exercise 2.

Build two identical columns in series:

1. Rename the copied `column` to `column_1`, both the variable and its quoted unit name `"column_1"`.
2. Add an identical unit `column_2` with the same settings.
3. Replace the direct connection `column_1` → `outlet` with `column_1` → `column_2` → `outlet`.
4. Copy the simulation and plotting lines from Exercise 1, adapted to your process name.

Predict the breakthrough time before you simulate.

```{tip}
`flow_sheet.remove_connection(column_1, outlet)` removes a connection; `add_unit()` and `add_connection()` work as in the copied setup.
```

In [ ]:
# Paste the model setup from Show code, then add column_2 in series


```{dropdown} Answer
Two identical columns double the liquid volume, so the front arrives at about 2 × 126 s ≈ 251 s. It is also broader: the tracer spreads in both columns.
```